<a href="https://colab.research.google.com/github/fishenzone/telegram_style/blob/main/tel_style.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {
        '2.10':'0.0.34',
        '2.9':'0.0.33.post1',
        '2.8':'0.0.32.post2'
    }.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install sentence-transformers scikit-learn pandas matplotlib seaborn -q

!git clone https://github.com/fishenzone/telegram_style.git


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use("dark_background")

%cd telegram_style

In [7]:
import sys
import random
import numpy as np
import torch

from pathlib import Path
from IPython.display import display

ROOT = Path("/content/telegram_style")
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from telegram_style.config import CHANNELS
from telegram_style.prompts import SYSTEM_PROMPTS, BASELINE_PROMPT
from telegram_style.io_utils import save_lines
from telegram_style.data_utils import (
    prepare_data_splits,
    load_saved_splits,
    build_dataset_from_pairs,
)
from telegram_style.model_utils import (
    load_base_model,
    attach_lora,
    generate_texts,
)
from telegram_style.train_utils import train_and_save
from telegram_style.memory_utils import cleanup, print_gpu, drop_vars
from telegram_style.metrics_utils import compute_metrics, print_interpretation, print_examples

r_seed = 66
random.seed(r_seed)
np.random.seed(r_seed)
torch.manual_seed(r_seed)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
splits = prepare_data_splits()

print("Prepared splits:")
print(f"type1 train={len(splits['type1']['train'])}, test={len(splits['type1']['test'])}")
print(f"type2 train={len(splits['type2']['train'])}, test={len(splits['type2']['test'])}")

splits = load_saved_splits()


Prepared splits:
type1 train=20, test=9
type2 train=20, test=11


In [9]:
model, tokenizer = load_base_model()
print("Loaded base model.")
print_gpu()


==((====))==  Unsloth 2026.3.5: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.59G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-14b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded base model.
GPU allocated: 10.37 GB | reserved: 10.41 GB


In [10]:
dataset_type1, train_texts_type1 = build_dataset_from_pairs(
    splits["type1"]["train"],
    SYSTEM_PROMPTS["type1"],
    tokenizer,
)
save_lines(train_texts_type1, CHANNELS["type1"]["formatted_train_path"])

dataset_type2, train_texts_type2 = build_dataset_from_pairs(
    splits["type2"]["train"],
    SYSTEM_PROMPTS["type2"],
    tokenizer,
)
save_lines(train_texts_type2, CHANNELS["type2"]["formatted_train_path"])

print(f"Prepared train dataset type1: {len(train_texts_type1)} rows")
print(f"Prepared train dataset type2: {len(train_texts_type2)} rows")


Prepared train dataset type1: 20 rows
Prepared train dataset type2: 20 rows


In [11]:
test_inputs_type1 = [x["input"] for x in splits["type1"]["test"]]
test_inputs_type2 = [x["input"] for x in splits["type2"]["test"]]

print("=== Baseline generation: Type 1 ===")
baseline_type1 = generate_texts(
    model=model,
    tokenizer=tokenizer,
    inputs=test_inputs_type1,
    system_prompt=BASELINE_PROMPT,
    max_new_tokens=CHANNELS["type1"]["max_new_tokens"],
)
save_lines(baseline_type1, CHANNELS["type1"]["baseline_path"])
print(f"Saved {CHANNELS['type1']['baseline_path']}")

print("=== Baseline generation: Type 2 ===")
baseline_type2 = generate_texts(
    model=model,
    tokenizer=tokenizer,
    inputs=test_inputs_type2,
    system_prompt=BASELINE_PROMPT,
    max_new_tokens=CHANNELS["type2"]["max_new_tokens"],
)
save_lines(baseline_type2, CHANNELS["type2"]["baseline_path"])
print(f"Saved {CHANNELS['type2']['baseline_path']}")


=== Baseline generation: Type 1 ===
[1/9] done
[2/9] done
[3/9] done
[4/9] done
[5/9] done
[6/9] done
[7/9] done
[8/9] done
[9/9] done
Saved baseline_type1.txt
=== Baseline generation: Type 2 ===
[1/11] done
[2/11] done
[3/11] done
[4/11] done
[5/11] done
[6/11] done
[7/11] done
[8/11] done
[9/11] done
[10/11] done
[11/11] done
Saved baseline_type2.txt


In [12]:
model = attach_lora(model)
print("LoRA attached for Type 1")
print_gpu()

trainer_type1, trainer_stats_1 = train_and_save(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset_type1,
    output_dir=CHANNELS["type1"]["lora_dir"],
    label="Type 1 (banki_oil)",
)


Unsloth 2026.3.5 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


LoRA attached for Type 1
GPU allocated: 10.73 GB | reserved: 10.75 GB


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/20 [00:00<?, ? examples/s]

GPU = Tesla T4. Max memory = 14.563 GB.
11.84 GB reserved before training Type 1 (banki_oil).
=== Training Type 1 (banki_oil) ===


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 3 | Total steps = 9
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 64,225,280 of 14,832,532,480 (0.43% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.343100
2,2.299800
3,2.331100
4,2.188000
5,2.128800
6,2.008800
7,1.746300
8,1.625000
9,1.451400


Type 1 (banki_oil) training runtime: 131.98 sec
Peak reserved memory = 12.553 GB
Peak reserved memory for training = 0.713 GB
Peak reserved memory % = 86.198%
Peak reserved memory for training % = 4.896%
Saved LoRA to lora_type1


In [13]:
print("=== Styled generation: Type 1 ===")
styled_type1 = generate_texts(
    model=model,
    tokenizer=tokenizer,
    inputs=test_inputs_type1,
    system_prompt=SYSTEM_PROMPTS["type1"],
    max_new_tokens=CHANNELS["type1"]["max_new_tokens"],
)
save_lines(styled_type1, CHANNELS["type1"]["outputs_path"])
print(f"Saved {CHANNELS['type1']['outputs_path']}")


=== Styled generation: Type 1 ===
[1/9] done
[2/9] done
[3/9] done
[4/9] done
[5/9] done
[6/9] done
[7/9] done
[8/9] done
[9/9] done
Saved outputs_type1.txt


In [14]:
drop_vars(globals(), [
    "trainer_type1",
    "trainer_stats_1",
    "model",
    "tokenizer",
    "baseline_type1",
    "baseline_type2",
    "styled_type1",
])

cleanup()
print("After cleanup:")
print_gpu()

model, tokenizer = load_base_model()
print("Reloaded fresh base model for Type 2.")
print_gpu()

model = attach_lora(model)
print("Attached fresh LoRA for Type 2.")
print_gpu()


After cleanup:
GPU allocated: 1.64 GB | reserved: 1.72 GB
==((====))==  Unsloth 2026.3.5: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

unsloth/qwen3-14b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Reloaded fresh base model for Type 2.
GPU allocated: 12.02 GB | reserved: 12.05 GB
Attached fresh LoRA for Type 2.
GPU allocated: 12.26 GB | reserved: 12.27 GB


In [15]:
trainer_type2, trainer_stats_2 = train_and_save(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset_type2,
    output_dir=CHANNELS["type2"]["lora_dir"],
    label="Type 2 (moscowach)",
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/20 [00:00<?, ? examples/s]

GPU = Tesla T4. Max memory = 14.563 GB.
13.48 GB reserved before training Type 2 (moscowach).
=== Training Type 2 (moscowach) ===


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 3 | Total steps = 9
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 64,225,280 of 14,832,532,480 (0.43% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.397800
2,2.426500
3,2.376200
4,2.233200
5,2.187200
6,2.019500
7,1.749300
8,1.658800
9,1.547500


Type 2 (moscowach) training runtime: 129.31 sec
Peak reserved memory = 14.018 GB
Peak reserved memory for training = 0.538 GB
Peak reserved memory % = 96.258%
Peak reserved memory for training % = 3.694%
Saved LoRA to lora_type2


In [16]:
print("=== Styled generation: Type 2 ===")
styled_type2 = generate_texts(
    model=model,
    tokenizer=tokenizer,
    inputs=test_inputs_type2,
    system_prompt=SYSTEM_PROMPTS["type2"],
    max_new_tokens=CHANNELS["type2"]["max_new_tokens"],
)
save_lines(styled_type2, CHANNELS["type2"]["outputs_path"])
print(f"Saved {CHANNELS['type2']['outputs_path']}")


=== Styled generation: Type 2 ===
[1/11] done
[2/11] done
[3/11] done
[4/11] done
[5/11] done
[6/11] done
[7/11] done
[8/11] done
[9/11] done
[10/11] done
[11/11] done
Saved outputs_type2.txt


In [17]:
results = compute_metrics(
    train_banki=splits["type1"]["train"],
    train_moscow=splits["type2"]["train"])

print_interpretation(results)

print("\nSUMMARY DF")
display(results["summary_df"].round(4))

print("\nSTRUCTURE DF")
display(results["structure_df"].round(3))

print("\nCROSS STYLE DF")
display(results["cross_df"].round(4))


METRICS EXPLANATION

1) Cosine similarity to held-out reference
   - Each generated post is compared with the real held-out post.
   - HIGHER = BETTER.

2) Cross-style similarity matrix
   - Generated Type1 should be closer to Reference Type1 than to Reference Type2.
   - Generated Type2 should be closer to Reference Type2 than to Reference Type1.
   - DIAGONAL should be higher.

3) Style gap before/after
   - Own-style similarity minus other-style similarity.
   - HIGHER = BETTER.

4) Marker compliance
   - Type1: @banki_oil rate and no-emoji rate.
   - Type2: leading-emoji rate.
   - HIGHER = BETTER.

5) Structure closeness to references
   - How close outputs are to reference average length and sentence count.
   - LOWER distance = BETTER.



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

,channel,cosine_before,cosine_after,cosine_gain,style_gap_before,style_gap_after,style_gap_gain,final_to_own_ref,final_to_other_ref,main_marker_before,main_marker_after,aux_marker_before,aux_marker_after,word_dist_before,word_dist_after,sent_dist_before,sent_dist_after
0,banki_oil,0.8073,0.8293,0.0220,0.3574,0.4048,0.0475,0.9189,0.5140,0.0,1.0,0.7778,1.0,19.1111,21.2222,1.5556,1.2222
1,moscowach,0.8620,0.8601,-0.0019,0.3380,0.3907,0.0528,0.9493,0.5585,0.0,1.0,NaN,NaN,17.7273,16.9091,0.5455,0.8182



STRUCTURE DF


,set,avg_words,avg_chars,avg_sentences
0,ref_type1,37.000,267.778,3.000
1,baseline_type1,17.889,124.667,1.444
2,styled_type1,15.778,111.222,1.778
3,ref_type2,33.091,224.273,2.091
4,baseline_type2,15.364,103.182,1.545
5,styled_type2,16.182,104.818,1.273



CROSS STYLE DF


,Reference Type1,Reference Type2
Generated Type1,0.9189,0.5140
Generated Type2,0.5585,0.9493


In [18]:
print_examples(
    test_type1=splits["type1"]["test"],
    test_type2=splits["type2"]["test"],
    results=results,
    limit=5,
)



EXAMPLES

--- TYPE 1 / Example 1 ---
INPUT (neutral):
США купили у России рекордное количество лаврового листа — примерно на $10 тысяч.

REFERENCE (real channel post):
США и РФ возродили торговлю. Американцы закупили у России рекордное количество лаврового листа, почти на $10 тысяч. @banki_oil

BEFORE LoRA (baseline):
США купили у России рекордное количество лаврового листа — почти на $10 тысяч.

AFTER LoRA (styled):
США купили у России рекордное количество лаврового листа — $10 тыс. Это больше, чем в прошлом году. @banki_oil

--- TYPE 1 / Example 2 ---
INPUT (neutral):
Депутат Тюменской облдумы Алексей Салмин заявил, что критика власти со стороны граждан свидетельствует об их недостаточной зрелости.

REFERENCE (real channel post):
Россияне не имеют права критиковать власть, потому что не доросли до этого умом, заявил депутат Тюменской облдумы Алексей Салмин. «Мне не нравился тот порядок, который был в 90-х. Но потом я стал взрослее, понял, что вся власть от Бога, поэтому надо смирить